# NB3 — Feature Engineering

Computes rolling statistics over each sensor signal and scales the resulting feature matrix. Key steps:

- **Block detection** — identifies continuous recording segments; rolling windows stay within blocks to avoid spanning day-boundary gaps
- **Baseline statistics** — per-sensor mean and std fitted on training rows for `bias_dev` normalization
- **Rolling features** — 8 statistics × 98 sensors × 2 window sizes = 1,568 feature columns
- **NaN fill** — warm-up NaNs filled using training-split column means (no leakage)
- **Scale** — `StandardScaler` fitted on training rows only, applied to all rows
- **Save** `data/features.parquet`, `data/scaler.joblib`, `data/feature_cols.json`

**Prerequisites:** Run NB1 → NB2 first.

In [1]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 80)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

WINDOW_SIZE       = 60    # short window (rows) — step/dropout/noise detection
MIN_PERIODS       = 30
WINDOW_SIZE_LONG  = 120   # long window (test-rows) — drift detection
# Rationale: test rows are ~20.7% of all rows, so 120 test rows ≈ 580 actual rows ≈
# one minimum-duration drift event (600 rows). At this window size, a test fault row
# near the end of a drift event has its window nearly filled with fault rows, giving
# ~100% fault signal. A 600-row window spans ~2900 actual rows — 5× too wide —
# diluting the 124 fault rows to only ~20% of the window.
MIN_PERIODS_LONG  = 60

df = pd.read_parquet(DATA_DIR / "wadi_faulted.parquet")
sensor_ref  = json.loads((DATA_DIR / "sensor_cols.json").read_text())
SENSOR_COLS = sensor_ref["sensor_cols"]

print(f"Input:  wadi_faulted.parquet")
print(f"Output: features.parquet  |  scaler.joblib  |  feature_cols.json")
print(f"Loaded: {df.shape}")
print(f"Sensor columns: {len(SENSOR_COLS)}")
print(f"\nLabel counts: { {int(k):int(v) for k,v in df['label'].value_counts().sort_index().items()} }")


Input:  wadi_faulted.parquet
Output: features.parquet  |  scaler.joblib  |  feature_cols.json
Loaded: (954399, 110)
Sensor columns: 98

Label counts: {0: 757800, 1: 9977, 2: 186622}


## 1. Block Detection & Baseline Statistics

Two setup steps before rolling computation:

1. **Block detection** — timestamps more than 60 seconds apart start a new block; rolling windows are constrained within blocks to avoid blending signals across recording-day gaps
2. **Baseline statistics** — per-sensor mean and std fitted on all training rows (all labels), used to normalize `bias_dev` features

In [2]:
# Assign block IDs to the FULL dataset first (train + test combined).
# This is critical: if we compute block IDs per-split, isolated test
# windows appear as separate blocks and rolling windows never accumulate
# enough rows (min_periods=30), making nearly all test rolling features NaN.
# With full-dataset block IDs, all rows within a recording day share a block.
timestamps_sorted = df.sort_values("timestamp")["timestamp"]
time_diffs = timestamps_sorted.diff().dt.total_seconds().fillna(0)
is_new_block = time_diffs > WINDOW_SIZE  # gap > 60s → new block
df["block_id"] = is_new_block.cumsum().astype(int).reindex(df.index)

n_blocks = df["block_id"].nunique()
print(f"Total blocks (gaps > {WINDOW_SIZE}s): {n_blocks}")
print(f"Block row counts:")
for bid, grp in df.groupby("block_id"):
    print(f"  Block {bid:>2}: {len(grp):>7,} rows  "
          f"{grp['timestamp'].min().date()} → {grp['timestamp'].max().date()}")

# Per-sensor mean/std on all training rows (including attack/fault labels) — stable
# baseline that reflects the natural label mix in training data.
train_mask = df["split"] == "train"
train_sensor_means = df.loc[train_mask, SENSOR_COLS].mean().to_dict()
train_sensor_stds  = df.loc[train_mask, SENSOR_COLS].std().to_dict()
n_valid_bias = sum(1 for s in train_sensor_stds.values() if s > 1e-6)
print(f"\nTraining rows for baseline stats: {train_mask.sum():,}")
print(f"Sensors eligible for bias_dev (train std > 1e-6): {n_valid_bias}")

Total blocks (gaps > 60s): 14
Block row counts:
  Block  0:  21,583 rows  2017-09-25 → 2017-09-25
  Block  1:  86,141 rows  2017-09-26 → 2017-09-26
  Block  2:  86,056 rows  2017-09-27 → 2017-09-27
  Block  3:  86,134 rows  2017-09-28 → 2017-09-28
  Block  4:  55,044 rows  2017-09-29 → 2017-09-29
  Block  5:  26,358 rows  2017-10-02 → 2017-10-02
  Block  6:  86,142 rows  2017-10-03 → 2017-10-03
  Block  7:  85,942 rows  2017-10-04 → 2017-10-04
  Block  8:  86,175 rows  2017-10-05 → 2017-10-05
  Block  9:  86,176 rows  2017-10-06 → 2017-10-06
  Block 10:  76,270 rows  2017-10-07 → 2017-10-07
  Block 11:  21,575 rows  2017-10-09 → 2017-10-09
  Block 12:  86,190 rows  2017-10-10 → 2017-10-10
  Block 13:  64,613 rows  2017-10-11 → 2017-10-11

Training rows for baseline stats: 772,812
Sensors eligible for bias_dev (train std > 1e-6): 98


## 2. Rolling Feature Computation

Computes **8 statistics per sensor** over two backward-looking windows:

| Feature | Purpose |
|---|---|
| `mean`, `std`, `min`, `max` | Level and spread |
| `rate_of_change`, `slope` | Trend detection |
| `nan_count` | Dropout detection — computed on **raw, pre-fill** values |
| `bias_dev` | Normalized deviation from training baseline |

- **Short window (60 rows, ~4 s)** — captures step changes, dropouts, and noise spikes
- **Long window (120 rows)** — sized for drift: a minimum-length fault event (600 rows) contains ~124 test rows; at 120-row window, a late-event test row has ~100% fault signal in its window

In [3]:
def compute_rolling_features(
    df_split: pd.DataFrame,
    sensor_cols: list[str],
    window: int,
    min_periods: int,
    train_means: dict,
    train_stds: dict,
) -> pd.DataFrame:
    """
    Compute 8 rolling statistics per sensor using full-dataset block IDs.
    Groups by block_id so test rows within the same recording day accumulate
    a proper rolling window rather than being isolated per-split blocks.
    nan_count is computed on the raw (un-filled) series to capture dropouts.
    """
    from numpy.lib.stride_tricks import sliding_window_view

    _x = np.arange(window, dtype=np.float64) - (window - 1) / 2.0
    _Sxx = float((_x ** 2).sum())
    _sw = (_x / _Sxx).astype(np.float64)

    def _slope_kernel(s: np.ndarray) -> float:
        m = len(s)
        if m < 2:
            return np.nan
        if m == window:
            return float(np.dot(_sw, s))
        x = np.arange(m, dtype=np.float64) - (m - 1) / 2.0
        Sxx = float((x ** 2).sum())
        return 0.0 if Sxx == 0 else float(np.dot(x / Sxx, s))

    feature_frames = []

    for col in sensor_cols:
        series = df_split[col].astype("float64")

        mean = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).mean()
        ).rename(f"{col}__mean_{window}s")
        std = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).std()
        ).rename(f"{col}__std_{window}s")
        mn = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).min()
        ).rename(f"{col}__min_{window}s")
        mx = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).max()
        ).rename(f"{col}__max_{window}s")
        roc = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).apply(
                lambda w: (w[-1] - w[0]) / window, raw=True
            )
        ).rename(f"{col}__roc_{window}s")
        slope = series.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=min_periods).apply(
                _slope_kernel, raw=True
            )
        ).rename(f"{col}__slope_{window}s")

        # nan_count on raw (un-filled) series — captures intermittent_dropout
        nan_indicator = series.isna().astype("float64")
        nan_count = nan_indicator.groupby(df_split["block_id"]).transform(
            lambda x: x.rolling(window=window, min_periods=1).sum()
        ).rename(f"{col}__nan_count_{window}s")

        feature_frames.extend([mean, std, mn, mx, roc, slope, nan_count])

        # bias_dev: (rolling_mean − train_mean) / train_std
        t_std = train_stds.get(col, 0.0)
        if t_std is not None and t_std > 1e-6:
            t_mean = train_means.get(col, 0.0)
            bias_dev = pd.Series(
                (mean.values - t_mean) / t_std,
                index=mean.index,
                name=f"{col}__bias_dev_{window}s",
                dtype="float32",
            )
            feature_frames.append(bias_dev)

    return pd.concat(feature_frames, axis=1).astype("float32")

print("Rolling feature function defined.")


Rolling feature function defined.


Features computed per split to keep train and test rolling statistics separate. Can take ~15–20 min.

In [4]:
import gc

feature_splits = []

for split in ["train", "test"]:
    print(f"\nProcessing {split} split...")
    df_split = df[df["split"] == split].sort_values("timestamp").reset_index(drop=True)
    print(f"  Rows: {len(df_split):,}")

    feat_short = compute_rolling_features(
        df_split, SENSOR_COLS, WINDOW_SIZE, MIN_PERIODS,
        train_sensor_means, train_sensor_stds
    )
    feat_long = compute_rolling_features(
        df_split, SENSOR_COLS, WINDOW_SIZE_LONG, MIN_PERIODS_LONG,
        train_sensor_means, train_sensor_stds
    )
    feat_df = pd.concat([feat_short, feat_long], axis=1)

    fault_cols = [c for c in df_split.columns if c.startswith("fault_")]
    meta_cols  = ["timestamp", "split", "label", "block_id"] + fault_cols

    out_df = pd.concat([
        df_split[meta_cols].reset_index(drop=True),
        pd.DataFrame(df_split[SENSOR_COLS].values, columns=SENSOR_COLS),
        feat_df.reset_index(drop=True),
    ], axis=1)

    feature_splits.append(out_df)
    print(f"  Feature columns (short+long): {len(feat_df.columns)}")
    gc.collect()

df_features = pd.concat(feature_splits, ignore_index=True)
print(f"\nFull feature frame: {df_features.shape}")


Processing train split...
  Rows: 772,812
  Feature columns (short+long): 1568

Processing test split...
  Rows: 181,585
  Feature columns (short+long): 1568

Full feature frame: (954397, 1679)


## 3. NaN Fill & Feature Assembly

Rolling features produce NaN values during each block's warm-up period. Fill strategy:

- **Primary fill** — training-split column means (no test leakage)
- **Secondary fill** — 0 for any column whose entire training split is NaN
- **Raw sensor NaN** — sensor dropout columns filled with training means, then 0

In [5]:
ROLLING_COLS = [c for c in df_features.columns
                if c.endswith(("_60s", "_120s")) and c not in SENSOR_COLS]

print(f"Rolling feature columns: {len(ROLLING_COLS)}")
print(f"NaN before fill: {df_features[ROLLING_COLS].isna().sum().sum():,}")

# Primary fill: use train-split column means (no leakage from test)
train_fill_means = df_features[df_features["split"] == "train"][ROLLING_COLS].mean()
df_features[ROLLING_COLS] = df_features[ROLLING_COLS].fillna(train_fill_means)

# Secondary fill with 0 for any column whose entire training split was NaN
remaining = df_features[ROLLING_COLS].isna().sum().sum()
if remaining > 0:
    cols_still_nan = [c for c in ROLLING_COLS if df_features[c].isna().any()]
    print(f"  {remaining:,} NaN remain in {len(cols_still_nan)} columns after primary fill "
          f"— filling with 0")
    df_features[ROLLING_COLS] = df_features[ROLLING_COLS].fillna(0)

# Raw sensor NaN fill
raw_nan = df_features[SENSOR_COLS].isna().sum().sum()
if raw_nan > 0:
    raw_fill_means = df_features[df_features["split"] == "train"][SENSOR_COLS].mean()
    df_features[SENSOR_COLS] = df_features[SENSOR_COLS].fillna(raw_fill_means)
    df_features[SENSOR_COLS] = df_features[SENSOR_COLS].fillna(0)
    print(f"Filled {raw_nan:,} raw sensor NaNs")

FEATURE_COLS = SENSOR_COLS + ROLLING_COLS
print(f"\nTotal feature columns: {len(FEATURE_COLS)}")
print(f"  Raw:     {len(SENSOR_COLS)}")
print(f"  Rolling: {len(ROLLING_COLS)}  (60s window: ~{len(ROLLING_COLS)//2}, 120s window: ~{len(ROLLING_COLS)//2})")

# Check NaN/inf column-by-column to avoid materializing the full array
nan_count  = sum(df_features[c].isna().sum() for c in FEATURE_COLS)
inf_count  = sum(np.isinf(df_features[c].values).sum() for c in FEATURE_COLS)
assert nan_count == 0,  f"NaN values remain: {nan_count}"
assert inf_count == 0,  f"Inf values remain: {inf_count}"
print("Feature matrix clean (no NaN, no inf)")

Rolling feature columns: 1568
NaN before fill: 2,447,852
Filled 48,320 raw sensor NaNs

Total feature columns: 1666
  Raw:     98
  Rolling: 1568  (60s window: ~784, 120s window: ~784)
Feature matrix clean (no NaN, no inf)


## 4. Scale — Fit on Train, Apply to All (~5 min)

Fits a `StandardScaler` on training rows and applies it to all rows. Three memory-efficient steps:

- **Zero-variance detection** — drops sensors that are flat in a 100k-row training sample
- **Partial fit** — scaler fitted in 50k-row chunks to avoid materializing the full ~8 GB training matrix
- **Column-by-column transform** — peak memory ~3 MB per column

In [6]:
# Detect and drop zero-variance sensors using a 100k training sample (cheap).
train_mask = df_features["split"] == "train"
train_idx  = df_features.index[train_mask]

X_sample = df_features.loc[train_idx[:100_000], FEATURE_COLS].values.astype(np.float32)
scaler_detect = StandardScaler()
scaler_detect.fit(X_sample)
del X_sample

zero_std_base_sensors = set()
for i, std in enumerate(scaler_detect.scale_):
    if std < 1e-6:
        base = FEATURE_COLS[i].split("__")[0]
        zero_std_base_sensors.add(base)

zero_std_cols = [c for c in FEATURE_COLS if c.split("__")[0] in zero_std_base_sensors]
FEATURE_COLS  = [c for c in FEATURE_COLS if c not in zero_std_cols]
SENSOR_COLS   = [c for c in SENSOR_COLS  if c not in zero_std_base_sensors]
ROLLING_COLS  = [c for c in ROLLING_COLS if c not in zero_std_cols]

print(f"Zero-variance sensors dropped ({len(zero_std_base_sensors)}): {sorted(zero_std_base_sensors)}")
print(f"Features dropped: {len(zero_std_cols)}")
print(f"Remaining features: {len(FEATURE_COLS)}  (raw={len(SENSOR_COLS)}, rolling={len(ROLLING_COLS)})")

# Fit scaler using partial_fit on training data in 50k-row chunks.
# Avoids materializing a full training matrix (~8 GB with doubled feature count).
SCALE_CHUNK = 50_000
scaler = StandardScaler()
for start in range(0, len(train_idx), SCALE_CHUNK):
    chunk = df_features.loc[train_idx[start:start + SCALE_CHUNK], FEATURE_COLS].values.astype(np.float32)
    scaler.partial_fit(chunk)
    del chunk

# Transform in-place column-by-column — peak memory = one column at a time (~3 MB).
for i, col in enumerate(FEATURE_COLS):
    df_features[col] = (
        (df_features[col].values.astype(np.float32) - np.float32(scaler.mean_[i]))
        / np.float32(scaler.scale_[i])
    ).astype(np.float32)

# Verify on a small sample — avoid materializing full array
check = df_features.loc[train_idx[:10_000], FEATURE_COLS].values.astype(np.float32)
print(f"\nScaler fit on {train_mask.sum():,} training rows (partial_fit, {SCALE_CHUNK:,}-row chunks)")
print(f"Applied to {len(df_features):,} total rows (column-by-column, peak ~3 MB/col)")
print(f"Train feature mean (should be ~0): {check.mean():.4f}")
print(f"Train feature std  (should be ~1): {check.std():.4f}")
del check

Zero-variance sensors dropped (9): ['1_MV_002_STATUS', '1_MV_003_STATUS', '1_P_006_STATUS', '2A_AIT_002_PV', '2_FIC_101_CO', '2_FIC_301_SP', '2_MV_101_STATUS', '2_PIC_003_SP', '3_AIT_001_PV']
Features dropped: 153
Remaining features: 1513  (raw=89, rolling=1424)

Scaler fit on 772,812 training rows (partial_fit, 50,000-row chunks)
Applied to 954,397 total rows (column-by-column, peak ~3 MB/col)
Train feature mean (should be ~0): -0.4119
Train feature std  (should be ~1): 1.4038


## 5. Save

Writes three artifacts to `data/`:

- **`features.parquet`** — scaled feature matrix with `timestamp`, `split`, `label`, `fault_type`, and all feature columns
- **`scaler.joblib`** — fitted `StandardScaler` for inverse transforms or reuse in NB4
- **`feature_cols.json`** — feature column lists and window configuration consumed by NB4

In [7]:
# Save feature parquet
feat_cols_keep = ["timestamp","split","label"] + FEATURE_COLS
# Also keep fault_type for per-fault evaluation in NB4
if "fault_type" in df_features.columns:
    feat_cols_keep = ["timestamp","split","label","fault_type"] + FEATURE_COLS

out_path = DATA_DIR / "features.parquet"
df_features[feat_cols_keep].to_parquet(out_path, index=False)
print(f"Saved: {out_path}  shape={df_features[feat_cols_keep].shape}")

joblib.dump(scaler, DATA_DIR / "scaler.joblib")
print(f"Saved: {DATA_DIR / 'scaler.joblib'}")

(DATA_DIR / "feature_cols.json").write_text(json.dumps({
    "sensor_cols":      SENSOR_COLS,
    "rolling_cols":     ROLLING_COLS,
    "feature_cols":     FEATURE_COLS,
    "window_size":      WINDOW_SIZE,
    "window_size_long": WINDOW_SIZE_LONG,
    "min_periods":      MIN_PERIODS,
    "n_features":       len(FEATURE_COLS),
}, indent=2))
print(f"Saved: {DATA_DIR / 'feature_cols.json'}")

print(f"Completed: {datetime.now()}")


Saved: data/features.parquet  shape=(954397, 1517)
Saved: data/scaler.joblib
Saved: data/feature_cols.json
Completed: 2026-04-24 15:25:11.685849
